# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey dataset using the `mlcroissant` library in Python. We walk through metadata discovery, extraction, and typical exploration/processing steps using only entity `@id` references for maximal reproducibility.

### Dataset Source
The dataset is described by a Croissant schema accessible at the following URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

> **Note:** All entities (record sets, fields, columns) are referenced by their `@id` values, in accordance with Croissant and FAIR data standards.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing the metadata object
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("License:", metadata.license)
print("Version:", metadata.version)
print("Published:", metadata.datePublished)
print("Keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview

Review the available record sets, the fields each set contains, and all associated Croissant `@id` items. These IDs will be used to reference entities for extraction and filtering.

In [ ]:
# Fetch all record sets in the Croissant schema
record_sets = [rs for rs in dataset.metadata.record_sets]

print("Number of record sets:", len(record_sets))

for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1}: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else '[no description]'}")
    fields = getattr(record_set, 'fields', [])
    if fields:
        print("  Fields (with @id):")
        for field in fields:
            print(f"   - {field.name} (@id: {field.id}) [type: {getattr(field, 'data_type', 'unknown type')}]" )
    else:
        print("  No fields found.")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will extract all record sets and load them into pandas DataFrames, using each record set's `@id`.


In [ ]:
# Build a dictionary {record_set_id: DataFrame}

rs_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id} with {len(df)} records and columns: {list(df.columns)}")
    else:
        print(f"No records found for record set {rs_id}.")

# Select the primary tabular record set (by convention, usually the first with rows)
main_rs_id = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        main_rs_id = rs_id
        break

if main_rs_id:
    print(f"Using main record set: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering records, normalizing numeric fields, and grouping. All field IDs must be referenced using the Croissant `@id` system. 

For demonstration, we'll select fields such as GAD-7, PHQ-9, or any available numeric score from the metadata above.

In [ ]:
# Example: Identify a numeric field in the primary record set
# (Edit as needed depending on the actual IDs from the overview)

# We'll use field @id reference (for example 'GAD_7_Score'), please replace with true @id as needed
main_df = dataframes[main_rs_id]
pp = pprint.PrettyPrinter()

# Try to guess likely numeric columns for demonstration
numeric_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['score', 'gad', 'phq', 'psq', 'age', 'numeric', 'number', 'symptom'])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # For demonstration
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field found. Please check available columns.")

# Filtering records where chosen numeric_field > threshold
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a demographic or category field, e.g., 'gender', 'marital_status', etc.
group_candidates = [c for c in main_df.columns if c.lower() in ['gender', 'sex', 'marital_status', 'education', 'village', 'level_of_education']]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df)
else:
    print("No group field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group if possible
if group_candidates:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} vs {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load, inspect, and explore the FAIR² mental health survey dataset for Kilifi County, Kenya. We discovered available record sets, extracted them using their Croissant `@id`, performed basic EDA and normalization, and visualized some distributions. This approach, referencing only `@id` fields, ensures reproducibility and alignment with the FAIR principles for data interoperability.

**Next steps could include:**
- Advanced statistical analysis or machine learning (predicting mental health scores).
- Integration with other datasets using Croissant semantics.
- Programmatic metadata discovery and schema-driven data validation.
